# Biohub Cell Tracking — Kaggle Submission Notebook

Runs inference on the hidden test set and writes `submission.csv`.

**Inputs expected:**
- `/kaggle/input/biohub-cell-tracking-during-development/test/` — test .zarr volumes
- `/kaggle/input/<your-model-dataset>/best.pt` — trained checkpoint (automatically resolved)
- `/kaggle/input/<your-model-dataset>/src/` — source package (automatically resolved)

**Output:** `/kaggle/working/submission.csv`

In [ ]:
# ── Install / upgrade dependencies ──────────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('zarr>=3.0', 'scipy>=1.10', 'pandas>=2.0', 'tqdm', 'scikit-image>=0.21')

In [ ]:
# ── Paths & Environment Setup ───────────────────────────────────────────────────
import sys
from pathlib import Path

OUTPUT_CSV    = Path('/kaggle/working/submission.csv')

# Ensure /kaggle/working is in path
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# Fast path resolution: Scan immediate subdirectories of /kaggle/input and skip the competition dataset
input_root = Path("/kaggle/input")
CKPT_PATH = None
TEST_ROOT = None

if input_root.exists():
    search_paths = [input_root]
    competitions_dir = input_root / "competitions"
    if competitions_dir.exists():
        search_paths.append(competitions_dir)
        
    for parent in search_paths:
        for child in parent.iterdir():
            if "biohub-cell-tracking-during-development" in child.name:
                # Search for test Zarrs
                candidates = []
                for pattern in ["*.zarr", "*/*.zarr", "*/*/*.zarr"]:
                    candidates.extend(list(child.glob(pattern)))
                if candidates:
                    test_candidates = [p for p in candidates if "test" in p.parts or "test" in str(p)]
                    if test_candidates:
                        TEST_ROOT = test_candidates[0].parent
                    else:
                        TEST_ROOT = candidates[0].parent
                continue  # Bypassed the massive competition directory recursively
                
            if child.is_dir():
                # Search for src
                if child.name == "src":
                    if str(child.parent) not in sys.path:
                        sys.path.insert(0, str(child.parent))
                        print(f"Added source directory to sys.path: {child.parent}")
                else:
                    for p in child.glob("src"):
                        if p.is_dir() and str(child) not in sys.path:
                            sys.path.insert(0, str(child))
                            print(f"Added source directory to sys.path: {child}")
                
                # Search for best.pt
                for p in child.rglob("best.pt"):
                    if p.is_file():
                        CKPT_PATH = p
                        break

if TEST_ROOT is None:
    TEST_ROOT = Path('/kaggle/input/biohub-cell-tracking-during-development/test')

if CKPT_PATH is None:
    # Fallback to local working directory
    CKPT_PATH = Path('/kaggle/working/best.pt')

print(f'Test root  : {TEST_ROOT}  (exists={TEST_ROOT.exists()})')
print(f'Checkpoint : {CKPT_PATH}  (exists={CKPT_PATH.exists()})')

In [ ]:
# ── Build config programmatically ───────────────────────────────────────────────
from src.config import Config, ModelConfig, DataConfig, TrainingConfig, InferenceConfig

config = Config(
    model=ModelConfig(
        in_channels=1,
        base_features=24,
        downsample_factors=[[1, 2, 2], [2, 2, 2], [2, 2, 2]],
        nms_kernel=5,
        nms_threshold=0.4,
    ),
    data=DataConfig(
        data_root=str(TEST_ROOT),
        input_shape=[64, 256, 256],
        scale=[1.625, 0.40625, 0.40625],
        normalize_mode='percentile',
    ),
    inference=InferenceConfig(
        checkpoint=str(CKPT_PATH),
        max_link_distance_um=7.0,
        output=str(OUTPUT_CSV),
    ),
)
print('Config ready.')

In [ ]:
# ── Load model ───────────────────────────────────────────────────────────────────
import torch
from src.infer import load_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = load_model(config, device)
print(f'Model output shape: {model.output_shape}')

In [ ]:
# ── Run inference ────────────────────────────────────────────────────────────────
from src.infer import run_inference

df = run_inference(config, data_root=str(TEST_ROOT))
print(f'Total rows: {len(df):,}   (nodes + edges)')
df.head(10)

In [ ]:
# ── Validate submission format ───────────────────────────────────────────────────
REQUIRED_COLS = ['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id']
assert list(df.columns) == REQUIRED_COLS, f'Column mismatch: {list(df.columns)}'

node_rows = df[df.row_type == 'node']
edge_rows = df[df.row_type == 'edge']
print(f'Nodes : {len(node_rows):,}')
print(f'Edges : {len(edge_rows):,}')
print(f'Datasets in submission: {df.dataset.nunique()}')

# Node rows should have valid (non-negative) coordinates
assert (node_rows[['t','z','y','x']] >= 0).all().all(), 'Negative node coordinates!'
# Edge rows must have valid source/target IDs
assert (edge_rows[['source_id','target_id']] > 0).all().all(), 'Invalid edge IDs!'

print('All checks passed.')

In [ ]:
# ── Save submission ───────────────────────────────────────────────────────────────
df.to_csv(str(OUTPUT_CSV), index=False)
print(f'submission.csv written → {OUTPUT_CSV}  ({OUTPUT_CSV.stat().st_size / 1024:.1f} KB)')